In [ ]:
import os
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

Base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv("GOOGLE_API_KEY")
Model = "gemini-3.6-flash"
myAi = OpenAI(base_url=Base_url,api_key=api_key)

ticket_price = {
    # India
    "delhi": "$120",
    "mumbai": "$150",
    "bangalore": "$180",
    "bengaluru": "$180",
    "hyderabad": "$170",
    "chennai": "$165",
    "kolkata": "$155",
    "pune": "$175",
    "ahmedabad": "$145",
    "jaipur": "$130",
    "lucknow": "$135",
    "goa": "$99",
    "kochi": "$190",
    "chandigarh": "$140",
    "indore": "$160",
    "bhubaneswar": "$175",
    "patna": "$150",
    "srinagar": "$200",
    "varanasi": "$145",

    # Europe
    "london": "$750",
    "paris": "$899",
    "rome": "$820",
    "madrid": "$850",
    "barcelona": "$870",
    "berlin": "$830",
    "amsterdam": "$880",
    "vienna": "$810",
    "prague": "$790",
    "zurich": "$950",
    "lisbon": "$900",
    "athens": "$840",
    "istanbul": "$700",
    "dublin": "$780",
    "brussels": "$860",
    "copenhagen": "$890",
    "stockholm": "$920",
    "oslo": "$940",
    "helsinki": "$960",

    # North America
    "new york": "$1100",
    "los angeles": "$1250",
    "san francisco": "$1300",
    "chicago": "$1150",
    "boston": "$1120",
    "washington": "$1140",
    "miami": "$1200",
    "seattle": "$1280",
    "las vegas": "$1230",
    "toronto": "$1050",
    "vancouver": "$1150",
    "montreal": "$1080",

    # Asia
    "tokyo": "$1400",
    "osaka": "$1350",
    "kyoto": "$1380",
    "seoul": "$1200",
    "beijing": "$1050",
    "shanghai": "$1100",
    "hong kong": "$850",
    "singapore": "$650",
    "bangkok": "$550",
    "kuala lumpur": "$580",
    "jakarta": "$620",
    "manila": "$700",
    "taipei": "$900",
    "dubai": "$199",
    "abu dhabi": "$230",
    "doha": "$350",
    "riyadh": "$400",
    "jeddah": "$420",
    "muscat": "$300",

    # Middle East
    "tel aviv": "$650",
    "amman": "$680",
    "beirut": "$700",
    "kuwait city": "$450",
    "bahrain": "$420",

    # Africa
    "cairo": "$600",
    "cape town": "$850",
    "johannesburg": "$800",
    "nairobi": "$650",
    "lagos": "$750",
    "casablanca": "$780",
    "addis ababa": "$620",
    "mauritius": "$500",

    # Australia & Oceania
    "sydney": "$1200",
    "melbourne": "$1180",
    "brisbane": "$1250",
    "perth": "$1100",
    "adelaide": "$1190",
    "auckland": "$1300",
    "wellington": "$1350",

    # South America
    "sao paulo": "$1400",
    "rio de janeiro": "$1450",
    "buenos aires": "$1500",
    "lima": "$1350",
    "santiago": "$1420",
    "bogota": "$1300",
    "quito": "$1380",

    # Popular tourist destinations
    "maldives": "$450",
    "bali": "$600",
    "phuket": "$580",
    "seychelles": "$700",
    "santorini": "$850",
    "venice": "$830",
    "milan": "$840",
    "nice": "$900",
    "hawaii": "$1350",
    "orlando": "$1200",
    "mexico city": "$1250",
    "cancun": "$1300"
}

def get_ticket_price(destination_city):
    price = ticket_price.get(destination_city.lower(),'Unknown Price')
    return f"The Price for the Flight of {destination_city} is {price}"

tool_schema = [{
    "type": "function",
    "function": {
        "name": "get_ticket_price",
        "parameters": {
            "type": "object",
            "properties": {
                "destination_city": {
                    "type": "string"
                }
            },
            "required": ["destination_city"]
        }
    }
}]

def handle_tools(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role":'tool',
                'content':price_details,
                'tool_call_id':tool_call.id
            })
        return responses
    
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

def chat(message,history):
    history = [{'role':h['role'],'content':h['content']} for h in history]
    messages = [{'role':'system','content':system_message}]+history+[{'role':'user','content':message}]
    response = myAi.chat.completions.create(model = Model,messages = messages,tools = tool_schema)

    while response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        responses = handle_tools(message)
        messages.append(message)
        messages.extend(responses)
        response = myAi.chat.completions.create(model=Model,messages=messages,tools=tool_schema)
    
    return response.choices[0].message.content

gr.ChatInterface(fn = chat,save_history = True).launch(share = True,inline = False,inbrowser = True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://6accc5b0a9f88328b8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
